In [1]:
# using existing code from https://medium.com/algorithmic-trading/advanced-high-frequency-trading-strategy-leveraging-order-book-imbalance-and-vwap-for-enhanced-74b93233b6a7
# appplying only OBI strategy
# try to use polars library for faster processing
# %pip install polars

In [2]:
import pandas as pd
import numpy as np
import polars as pl

In [3]:
df = pl.read_csv("./data/id9o0mbu0rvqqbuf.csv")
df.shape

(11590088, 14)

In [4]:
# time = df['TIME_M']
# time.str.to_time(format="%H:%M:%S%.f")

In [5]:
# df['TIME_M'] = df['TIME_M'].apply(lambda x: f"{x}.000000" if '.' not in x else x)

# # Convert TIME_M to datetime with nanosecond precision and update the column type
# df['TIME_PD'] = pd.to_datetime(df['TIME_M'], format='%H:%M:%S.%f', errors='coerce')
# df.shape
# df.dtypes

In [6]:
df.head()

DATE,TIME_M,EX,BID,BIDSIZ,ASK,ASKSIZ,QU_COND,QU_SEQNUM,NATBBO_IND,QU_CANCEL,QU_SOURCE,SYM_ROOT,SYM_SUFFIX
str,str,str,f64,i64,f64,i64,str,i64,i64,str,str,str,str
"""2023-05-10""","""3:59:00.005197350""","""K""",0.0,0,0.0,0,"""L""",1090,1,null,"""N""","""AAPL""",null
"""2023-05-10""","""3:59:00.095617932""","""Z""",0.0,0,0.0,0,"""L""",2024,1,null,"""N""","""AAPL""",null
"""2023-05-10""","""4:00:00.003953823""","""Q""",171.23,4,0.0,0,"""R""",3892,2,null,"""N""","""AAPL""",null
"""2023-05-10""","""4:00:00.003974778""","""Q""",171.23,4,171.44,4,"""R""",3893,4,null,"""N""","""AAPL""",null
"""2023-05-10""","""4:00:00.005169850""","""K""",171.02,10,0.0,0,"""Y""",3953,0,null,"""N""","""AAPL""",null


In [7]:
# Ensure TIME_M is parsed as a datetime column with strict=False to handle invalid formats
df = df.with_columns(
    pl.col("TIME_M").str.to_time(format="%H:%M:%S%.f").alias("TIME_M")
)

# Filter rows between 9:30:00 and 16:00:00
df = df.filter(
    pl.col("TIME_M").is_between(pl.time(9, 30), pl.time(16, 0))
    )

# Display the shape, first 5 rows, and last 5 rows
df.shape, df.head(5), df.tail(5)

((11517550, 14),
 shape: (5, 14)
 ┌────────────┬──────────────────┬─────┬────────┬───┬───────────┬───────────┬──────────┬────────────┐
 │ DATE       ┆ TIME_M           ┆ EX  ┆ BID    ┆ … ┆ QU_CANCEL ┆ QU_SOURCE ┆ SYM_ROOT ┆ SYM_SUFFIX │
 │ ---        ┆ ---              ┆ --- ┆ ---    ┆   ┆ ---       ┆ ---       ┆ ---      ┆ ---        │
 │ str        ┆ time             ┆ str ┆ f64    ┆   ┆ str       ┆ str       ┆ str      ┆ str        │
 ╞════════════╪══════════════════╪═════╪════════╪═══╪═══════════╪═══════════╪══════════╪════════════╡
 │ 2023-05-10 ┆ 09:30:00.0004316 ┆ Q   ┆ 172.98 ┆ … ┆ null      ┆ N         ┆ AAPL     ┆ null       │
 │            ┆ 16               ┆     ┆        ┆   ┆           ┆           ┆          ┆            │
 │ 2023-05-10 ┆ 09:30:00.0007351 ┆ U   ┆ 172.92 ┆ … ┆ null      ┆ N         ┆ AAPL     ┆ null       │
 │            ┆ 36               ┆     ┆        ┆   ┆           ┆           ┆          ┆            │
 │ 2023-05-10 ┆ 09:30:00.000890  ┆ P   ┆ 172.94 ┆

# should do some data exploration here, review some of the value counts to find cancelled orders etc

In [8]:
class NBBOTracker:
    """
    Tracks the National Best Bid and Offer (NBBO) across multiple exchanges from quote-level data.

    This class is intended to be used with high-frequency market data (e.g., TAQ) where each row
    represents a quote update from a specific exchange at a given timestamp. It maintains the
    current NBBO and accumulates size contributions at the best bid and ask across exchanges.

    Attributes
    ----------
    bid : float or None
        The current best bid price observed across exchanges.
    ask : float or None
        The current best ask price observed across exchanges.
    bid_sizes : dict
        Dictionary mapping exchange codes to bid sizes at the NBBO bid.
    ask_sizes : dict
        Dictionary mapping exchange codes to ask sizes at the NBBO ask.

    Methods
    -------
    update(df):
        Update NBBO state based on a new row of quote data. Adds the following columns to the DataFrame:
        - 'best_bid': the current best bid price
        - 'best_ask': the current best ask price
        - 'total_best_bid_size': cumulative bid size at NBBO bid across all exchanges
        - 'total_best_ask_size': cumulative ask size at NBBO ask across all exchanges
    """

    def __init__(self):
        self.bid = None
        self.ask = None
        self.bid_sizes = {}
        self.ask_sizes = {}

    def update(self, df: pl.DataFrame) -> pl.DataFrame:
        # Iterate over rows in the DataFrame
        updated_rows = []
        for row in df.iter_rows(named=True):
            exch = row["EX"]
            if row["NATBBO_IND"] == 4:  # NBBO update means the national best bid and offer have changed
                self.bid = row["BID"]
                self.ask = row["ASK"]
                self.bid_sizes = {exch: row["BIDSIZ"]} if row["BID"] is not None else {}
                self.ask_sizes = {exch: row["ASKSIZ"]} if row["ASK"] is not None else {}
            else:  # Otherwise, the national best bid and offer have not changed
                if row["BID"] is not None and row["BID"] == self.bid:
                    self.bid_sizes[exch] = row["BIDSIZ"]
                if row["ASK"] is not None and row["ASK"] == self.ask:
                    self.ask_sizes[exch] = row["ASKSIZ"]

            # Add NBBO-related columns to the row
            row["best_bid"] = self.bid
            row["best_ask"] = self.ask
            row["total_best_bid_size"] = sum(self.bid_sizes.values())
            row["total_best_ask_size"] = sum(self.ask_sizes.values())
            updated_rows.append(row)

        # Convert updated rows back to a polars DataFrame
        return pl.DataFrame(updated_rows)

In [9]:
tracker = NBBOTracker()
df = tracker.update(df)
df.head()

KeyboardInterrupt: 

In [ ]:
df[['total_best_bid_size', 'total_best_ask_size']].describe()

statistic,total_best_bid_size,total_best_ask_size
str,f64,f64
"""count""",1.151755e7,1.151755e7
"""null_count""",0.0,0.0
"""mean""",16.587392,16.982121
"""std""",12.888772,15.505426
"""min""",1.0,1.0
"""25%""",7.0,7.0
"""50%""",13.0,13.0
"""75%""",23.0,23.0
"""max""",324.0,754.0


### For 5 mins data in 10 May 2023
just taking a small sample, we have a max of 100 only. Would be good to see with full 24 hr dataset, what is the distribution

Lets see the sample of when size is more than 100

observed that OBI is a decent indicator of price movement, although the price only changed minimally
1. first signal flip at 10:03:36.187286990 is only .01 px change ~.5% return
2. next signal flip at 10:03:36.227561398 is .01 px change again

size here should be in a round lot (100 shares)

# Try OBI strategy

In [ ]:
class OBIVWAPStrategy:
    def __init__(self, vwap_window: int, obi_threshold: float, initial_cash: float = 100_000):
        self.vwap_window = vwap_window
        self.obi_threshold = obi_threshold
        self.cash = initial_cash
        self.position = 0
        self.account_balance = []

    def calculate_vwap(self, df: pl.DataFrame) -> pl.DataFrame:
        # Calculate MID_PRICE
        df = df.with_columns(
            ((pl.col("best_bid") + pl.col("best_ask")) / 2).alias("MID_PRICE")
        )
        # Calculate Volume
        df = df.with_columns(
            (pl.col("total_best_bid_size") + pl.col("total_best_ask_size")).alias("Volume")
        )
        # Calculate VWAP using rolling window
        df = df.with_columns(
            (
                (pl.col("MID_PRICE") * pl.col("Volume"))
                .rolling_sum(window_size=self.vwap_window)
                / pl.col("Volume").rolling_sum(window_size=self.vwap_window)
            ).alias("VWAP")
        )
        return df

    def calculate_obi(self, df: pl.DataFrame) -> pl.DataFrame:
        # Calculate Order Book Imbalance (OBI)
        df = df.with_columns(
            (
                (pl.col("total_best_bid_size") - pl.col("total_best_ask_size"))
                / (pl.col("total_best_bid_size") + pl.col("total_best_ask_size"))
            ).alias("OBI")
        )
        return df

    def generate_signals(self, df: pl.DataFrame) -> pl.DataFrame:
        # Calculate VWAP and OBI
        df = self.calculate_vwap(df)
        df = self.calculate_obi(df)

        # Generate signals based on OBI threshold
        df = df.with_columns(
            pl.when(pl.col("OBI") > self.obi_threshold)
            .then(1)  # Buy signal
            .when(pl.col("OBI") < -self.obi_threshold)
            .then(-1)  # Sell signal
            .otherwise(0)  # No signal
            .alias("Signal")
        )
        return df

    def backtest(self, df: pl.DataFrame) -> pl.DataFrame:
        # Initialize account balance tracking
        account_balance = []

        # Iterate over rows to simulate trading
        for row in df.iter_rows(named=True):
            if row["Signal"] == 1 and self.cash >= row["best_ask"] * 100 and self.position <= 1:
                # Buy 100 shares
                self.position = 100
                self.cash -= row["best_ask"] * 100
            elif row["Signal"] == -1 and self.position > -1:
                # Sell 100 shares
                self.position = -100
                self.cash += row["best_bid"] * 100
            elif row["Signal"] == 0 and self.position != 0:
                # Close position
                if self.position > 0:
                    self.cash += row["best_bid"] * self.position
                else:
                    self.cash -= row["best_ask"] * abs(self.position)
                self.position = 0

            # Record account balance
            account_balance.append(self.cash + self.position * row["MID_PRICE"])

        # Add account balance to the DataFrame
        df = df.with_columns(pl.Series("Account_Balance", account_balance))
        return df

In [ ]:
df['QU_CANCEL'].value_counts()

QU_CANCEL,count
null,u32
null,11517550


In [ ]:
strategy = OBIVWAPStrategy(vwap_window=50, obi_threshold=0.05)
signal_data = strategy.generate_signals(df)
backtest_data = strategy.backtest(signal_data)

print(backtest_data.head())


shape: (5, 24)
┌────────────┬─────────────────┬─────┬────────┬───┬──────┬───────────┬────────┬─────────────────┐
│ DATE       ┆ TIME_M          ┆ EX  ┆ BID    ┆ … ┆ VWAP ┆ OBI       ┆ Signal ┆ Account_Balance │
│ ---        ┆ ---             ┆ --- ┆ ---    ┆   ┆ ---  ┆ ---       ┆ ---    ┆ ---             │
│ str        ┆ time            ┆ str ┆ f64    ┆   ┆ f64  ┆ f64       ┆ i32    ┆ f64             │
╞════════════╪═════════════════╪═════╪════════╪═══╪══════╪═══════════╪════════╪═════════════════╡
│ 2023-05-10 ┆ 09:30:00.000431 ┆ Q   ┆ 172.98 ┆ … ┆ null ┆ -0.888889 ┆ -1     ┆ 99998.0         │
│ 2023-05-10 ┆ 09:30:00.000735 ┆ U   ┆ 172.92 ┆ … ┆ null ┆ -0.888889 ┆ -1     ┆ 99998.0         │
│ 2023-05-10 ┆ 09:30:00.000890 ┆ P   ┆ 172.94 ┆ … ┆ null ┆ -0.888889 ┆ -1     ┆ 99998.0         │
│ 2023-05-10 ┆ 09:30:00.000916 ┆ Q   ┆ 172.98 ┆ … ┆ null ┆ -0.837838 ┆ -1     ┆ 99998.0         │
│ 2023-05-10 ┆ 09:30:00.000934 ┆ P   ┆ 172.94 ┆ … ┆ null ┆ -0.837838 ┆ -1     ┆ 99998.0         │
└────

In [ ]:
backtest_data.tail(10)

DATE,TIME_M,EX,BID,BIDSIZ,ASK,ASKSIZ,QU_COND,QU_SEQNUM,NATBBO_IND,QU_CANCEL,QU_SOURCE,SYM_ROOT,SYM_SUFFIX,best_bid,best_ask,total_best_bid_size,total_best_ask_size,MID_PRICE,Volume,VWAP,OBI,Signal,Account_Balance
str,time,str,f64,i64,f64,i64,str,i64,i64,null,str,str,null,f64,f64,i64,i64,f64,i64,f64,f64,i32,f64
"""2023-05-10""",15:59:59.933502,"""U""",173.54,1,173.6,1,"""R""",83997338,0,null,"""N""","""AAPL""",null,173.55,173.56,2,40,173.555,42,173.551775,-0.904762,-1,7563624.5
"""2023-05-10""",15:59:59.944550,"""Q""",173.55,58,173.56,31,"""R""",83997486,4,null,"""N""","""AAPL""",null,173.55,173.56,58,31,173.555,89,173.551911,0.303371,1,7580979.5
"""2023-05-10""",15:59:59.980121,"""Q""",173.55,58,173.56,42,"""R""",83998027,4,null,"""N""","""AAPL""",null,173.55,173.56,58,42,173.555,100,173.552056,0.16,1,7580979.5
"""2023-05-10""",15:59:59.980148,"""X""",173.53,1,173.58,2,"""R""",83998032,0,null,"""N""","""AAPL""",null,173.55,173.56,58,42,173.555,100,173.552203,0.16,1,7580979.5
"""2023-05-10""",15:59:59.996143,"""M""",173.35,4,173.68,3,"""R""",83998324,0,null,"""N""","""AAPL""",null,173.55,173.56,58,42,173.555,100,173.552349,0.16,1,7580979.5
"""2023-05-10""",15:59:59.996241,"""V""",139.0,2,175.0,4,"""R""",83998329,0,null,"""N""","""AAPL""",null,173.55,173.56,58,42,173.555,100,173.552494,0.16,1,7580979.5
"""2023-05-10""",15:59:59.996299,"""K""",173.55,1,173.58,2,"""R""",83998331,0,null,"""N""","""AAPL""",null,173.55,173.56,59,42,173.555,101,173.552639,0.168317,1,7580979.5
"""2023-05-10""",15:59:59.996663,"""M""",173.35,4,173.56,4,"""R""",83998337,0,null,"""N""","""AAPL""",null,173.55,173.56,59,46,173.555,105,173.552789,0.12381,1,7580979.5
"""2023-05-10""",15:59:59.996674,"""M""",173.35,4,173.68,3,"""R""",83998339,0,null,"""N""","""AAPL""",null,173.55,173.56,59,46,173.555,105,173.55294,0.12381,1,7580979.5


In [ ]:
backtest_data["Account_Balance"].describe()

statistic,value
str,f64
"""count""",1.151755e7
"""null_count""",0.0
"""mean""",5.5046e6
"""std""",2.5895e6
"""min""",13154.0
"""25%""",3505157.5
"""50%""",5292896.5
"""75%""",7115278.5
"""max""",1.1852e7


In [ ]:
backtest_data['OBI'].sum()

13132.8734755557

In [ ]:
import plotly.express as px
px.line(backtest_data, x='QU_SEQNUM', y='Account_Balance', title='Account Balance Over Time')

In [ ]:
backtest_data.tail(5)

DATE,TIME_M,EX,BID,BIDSIZ,ASK,ASKSIZ,QU_COND,QU_SEQNUM,NATBBO_IND,QU_CANCEL,QU_SOURCE,SYM_ROOT,SYM_SUFFIX,best_bid,best_ask,total_best_bid_size,total_best_ask_size,MID_PRICE,Volume,VWAP,OBI,Signal,Account_Balance
str,time,str,f64,i64,f64,i64,str,i64,i64,null,str,str,null,f64,f64,i64,i64,f64,i64,f64,f64,i32,f64
"""2023-05-10""",15:59:59.996241,"""V""",139.0,2,175.0,4,"""R""",83998329,0,null,"""N""","""AAPL""",null,173.55,173.56,58,42,173.555,100,173.552494,0.16,1,7580979.5
"""2023-05-10""",15:59:59.996299,"""K""",173.55,1,173.58,2,"""R""",83998331,0,null,"""N""","""AAPL""",null,173.55,173.56,59,42,173.555,101,173.552639,0.168317,1,7580979.5
"""2023-05-10""",15:59:59.996663,"""M""",173.35,4,173.56,4,"""R""",83998337,0,null,"""N""","""AAPL""",null,173.55,173.56,59,46,173.555,105,173.552789,0.12381,1,7580979.5
"""2023-05-10""",15:59:59.996674,"""M""",173.35,4,173.68,3,"""R""",83998339,0,null,"""N""","""AAPL""",null,173.55,173.56,59,46,173.555,105,173.55294,0.12381,1,7580979.5
"""2023-05-10""",15:59:59.997234,"""Q""",173.55,58,173.56,76,"""R""",83998381,4,null,"""N""","""AAPL""",null,173.55,173.56,58,76,173.555,134,173.553107,-0.134328,-1,7563623.5


In [ ]:
import matplotlib.pyplot as plt

# Convert to pandas
pdf = df.to_pandas()

# Plot
plt.plot(pdf["date"], pdf["value"])
plt.xlabel("Date")
plt.ylabel("Value")
plt.title("Time Series Plot")
plt.show()